Cluster the 2018–2025 regular-season pitches produced by `dataset_creation.ipynb`. Compare K-means with k=4 and k=5 and visualize the standardized features with PCA. Saved outputs have been cleared for publication.


In [ ]:
from pyspark.sql import SparkSession, functions as F, Window as W
from pyspark.ml.feature import VectorAssembler, StandardScaler, PCA
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

In [ ]:
spark = (
    SparkSession.builder
    .master("local")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.parquet.columnarReaderBatchSize", "1024")
    .appName("Clustering")
    .getOrCreate()
)

In [ ]:
df = spark.read.parquet("./data/full_dataset.parquet")

In [ ]:
df.show(5)

In [ ]:
def add_strike_zone(df):
    ball_radius = 0.12
    half_plate = 17 / 24

    return df.withColumn(
        "in_strike_zone",
        F.when(
            F.col("plate_x").isNull() | F.col("plate_z").isNull() | F.col("sz_bot").isNull() | F.col("sz_top").isNull(),
            F.lit(None).cast("int"),
        ).when(
            (F.abs(F.col("plate_x")) <= half_plate + ball_radius)
            & (F.col("plate_z") >= F.col("sz_bot") - ball_radius)
            & (F.col("plate_z") <= F.col("sz_top") + ball_radius),
            1,
        ).otherwise(0),
    )

from pyspark.sql import functions as F

df = df.withColumn(
    "pfx_x_arm",
    F.when(F.col("p_throws") == "R", F.col("pfx_x"))
     .when(F.col("p_throws") == "L", -F.col("pfx_x"))
)

df = add_strike_zone(df)

In [ ]:
numerical = ["release_speed", "release_spin_rate", "spin_axis", "pfx_x_arm", "pfx_z", "release_extension", "release_pos_x", "release_pos_z"]

cluster_df = (
    df.select("pitch_type", *numerical, "in_strike_zone").dropna(subset=numerical)
)

In [ ]:
assembler = VectorAssembler(
    inputCols=numerical,
    outputCol="raw_features"
)

assembled_df = assembler.transform(cluster_df)

scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="scaled_features",
    withMean=True,
    withStd=True
)

scaler_model = scaler.fit(assembled_df)
data = scaler_model.transform(assembled_df).select(
    "pitch_type", "in_strike_zone", "release_speed", "pfx_x_arm", "pfx_z", "scaled_features"
).cache()
print("Clustering observations:", data.count())


In [ ]:
pca = PCA(
    inputCol="scaled_features",
    outputCol='pca_features',
    k=4)

pca_model = pca.fit(data)
pca_data = pca_model.transform(data)

In [ ]:
pca_data.select("pca_features").show(5, truncate=False)
print("explained variance:", pca_model.explainedVariance)

In [ ]:
from pyspark.ml.evaluation import ClusteringEvaluator

evaluator = ClusteringEvaluator(
    featuresCol="scaled_features",
    predictionCol="prediction",
    metricName="silhouette",
    distanceMeasure="squaredEuclidean"
)

models = {}
results = []

for k in range(4, 6):
    model = KMeans(
        k=k,
        seed=42,
        featuresCol="scaled_features",
        predictionCol="prediction",
        maxIter=30
    ).fit(data)

    models[k] = model

    predictions = model.transform(data)
    silhouette = evaluator.evaluate(predictions)

    results.append((k, silhouette))
    print(f"k={k} | Silhouette: {silhouette:.4f}")

In [ ]:
import matplotlib.pyplot as plt

ks = [r[0] for r in results]
scores = [r[1] for r in results]

plt.figure(figsize=(8, 5))
plt.plot(ks, scores, marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Silhouette score")
plt.title("KMeans — Silhouette by number of clusters")
plt.xticks(ks)
plt.grid(alpha=0.3)
plt.show()

In [ ]:

for k in [4, 5]:
    predictions = models[k].transform(data)

    composition = (
        predictions.groupBy("prediction", "pitch_type")
        .count()
        .withColumn(
            "cluster_total",
            F.sum("count").over(W.partitionBy("prediction"))
        )
        .withColumn(
            "percentage",
            F.round(F.col("count") / F.col("cluster_total") * 100, 2)
        )
        .orderBy("prediction", F.desc("percentage"))
    )

    print(f"\nKMeans k={k}")
    composition.show(100, truncate=False)

In [ ]:
clustered = models[5].transform(data)

cluster_stats = (
    clustered.groupBy("prediction")
    .agg(
        F.count("*").alias("count"),
        F.round(F.avg("release_speed"), 2).alias("avg_speed"),
        F.round(F.avg("pfx_x_arm"), 3).alias("avg_pfx_x"),
        F.round(F.avg("pfx_z"), 3).alias("avg_pfx_z"),
        F.count("in_strike_zone").alias("zone_observations"),
        F.sum("in_strike_zone").alias("strike_count"),
        F.round(F.avg("in_strike_zone") * 100, 2).alias("strike_percentage")
    )
    .orderBy("prediction")
)

# strike percentage measures geometric zone membership, not the umpire's call
cluster_stats.show(truncate=False)

### Cluster visualization on a sample

Select up to 5,000 pitches using a seeded random ordering in Spark. Apply the existing k=4 and k=5 models and plot the first three PCA components. Only the plotting columns are converted to pandas. Cluster IDs are local to each model; rare clusters may have few or no sampled points.


In [ ]:
import matplotlib.pyplot as plt
from pyspark.sql import functions as F
from pyspark.ml.functions import vector_to_array

sample_size = 5000

sampled_data = (
    pca_data
    .orderBy(F.rand(42))
    .limit(sample_size)
    .cache()
)

sampled_data.count()

fig = plt.figure(figsize=(16, 8))
colors = plt.get_cmap("tab10")

for column, k in enumerate([4, 5]):

    plot_df = (
        models[k].transform(sampled_data)
        .select("pca_features", "prediction")
        .withColumn("PC1", vector_to_array("pca_features")[0])
        .withColumn("PC2", vector_to_array("pca_features")[1])
        .withColumn("PC3", vector_to_array("pca_features")[2])
        .toPandas()
    )

    ax = fig.add_subplot(1, 2, column + 1, projection="3d")

    for cluster in range(k):
        points = plot_df[plot_df["prediction"] == cluster]

        ax.scatter(
            points["PC1"],
            points["PC2"],
            points["PC3"],
            s=12,
            alpha=0.85,
            color=colors(cluster),
            label=f"Cluster {cluster}",
            rasterized=True
        )

    ax.set_xlabel(f"PC1 ({pca_model.explainedVariance[0]:.2%})")
    ax.set_ylabel(f"PC2 ({pca_model.explainedVariance[1]:.2%})")
    ax.set_zlabel(f"PC3 ({pca_model.explainedVariance[2]:.2%})")

    ax.set_title(f"KMeans k={k} — 3D PCA")

    ax.view_init(elev=25, azim=45)
    ax.legend(markerscale=2, loc="upper right")
    ax.grid(alpha=0.3)

sampled_data.unpersist()

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

loadings = pd.DataFrame(
    pca_model.pc.toArray(),
    index=numerical,
    columns=[f"PC{i+1}" for i in range(pca_model.getK())]
)

display(loadings.round(3))

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

for k in [4, 5]:
    clustered = models[k].transform(data)

    counts = (
        clustered
        .groupBy("prediction", "pitch_type")
        .count()
    )

    window = Window.partitionBy("prediction")

    composition = (
        counts
        .withColumn("cluster_total", F.sum("count").over(window))
        .withColumn(
            "percentage",
            F.round(F.col("count") / F.col("cluster_total") * 100, 2)
        )
        .orderBy("prediction", F.desc("percentage"))
    )

    print(f"\nKMeans k={k}")
    composition.show(100, truncate=False)

In [ ]:
from pyspark.sql import functions as F
from pyspark.ml.functions import vector_to_array

physical_features = [
    "release_speed",
    "release_spin_rate",
    "spin_axis",
    "pfx_x_arm",
    "pfx_z",
    "release_extension",
    "release_pos_x",
    "release_pos_z"
]

for k in [4, 5]:
    clustered = (
        models[k]
        .transform(data)
        .withColumn("features_array", vector_to_array("scaled_features"))
    )

    cluster_stats = (
        clustered
        .groupBy("prediction")
        .agg(
            F.count("*").alias("n_pitches"),
            *[
                F.round(
                    F.avg(F.col("features_array")[i]), 3
                ).alias(feature)
                for i, feature in enumerate(physical_features)
            ]
        )
        .orderBy("prediction")
    )

    print(f"\nKMeans k={k}")
    cluster_stats.show(truncate=False)